In [1]:
# The City College of New York, GSOE
# Written by Hasan Suca Kayman and Prof. M. Umit Uyar and Khadija Begum
# September, 2026
# Algorithmic trading of financial stocks using ANNs

In [ ]:
import numpy as np
import pandas as pd
import time
from tensorflow import keras

keras.utils.set_random_seed(42)

#this is a utility function to be called later in the code:
def print_weights(weights):
    # weights = model.get_weights();
    print('\n******* WEIGHTS OF ANN *******\n')
    for i in range(int(len(weights)/2)):
        print('Weights W%d:\n' %(i), weights[i*2])
        print('Bias b%d:\n' %(i), weights[(i*2)+1])

#% DATA EXTRACTION
historical_data_file = 'MSFT.csv'
print('\n********* NOW READING SAMLE DATA USING ', historical_data_file,'*********')
time.sleep(3)

## load the historical stock data
df = pd.read_csv(historical_data_file) # (df stands for data frame)
df


********* NOW READING SAMLE DATA USING  MSFT.csv *********


,Date,Open,High,Low,Close,Adj Close,Volume
0,3/20/2019,117.389999,118.750000,116.709999,117.519997,115.985443,28113300
1,3/21/2019,117.139999,120.820000,117.089996,120.220001,118.650192,29854400
2,3/22/2019,119.500000,119.589996,117.040001,117.050003,115.521576,33624500
3,3/25/2019,116.559998,118.010002,116.320000,117.660004,116.123627,27067100
4,3/26/2019,118.620003,118.709999,116.849998,117.910004,116.370346,26097700
...,...,...,...,...,...,...,...
249,3/16/2020,140.000000,149.350006,135.000000,135.419998,135.419998,87905900
250,3/17/2020,140.000000,147.500000,135.000000,146.570007,146.570007,81059800
251,3/18/2020,138.000000,146.000000,135.020004,140.399994,140.399994,81593200
252,3/19/2020,142.770004,150.149994,139.000000,142.710007,142.710007,85801400


In [ ]:
# define technical indicators for alogirhtmic trading:
short_window = 10
long_window = 30
## calculate the Simple Moving Average and add it to a new column in df;
## rolling means moving window starting from the current entry and
## the last 5 days
df['SMA'] = df['Close'].rolling(short_window).mean()
df

,Date,Open,High,Low,Close,Adj Close,Volume,SMA
0,3/20/2019,117.389999,118.750000,116.709999,117.519997,115.985443,28113300,NaN
1,3/21/2019,117.139999,120.820000,117.089996,120.220001,118.650192,29854400,NaN
2,3/22/2019,119.500000,119.589996,117.040001,117.050003,115.521576,33624500,NaN
3,3/25/2019,116.559998,118.010002,116.320000,117.660004,116.123627,27067100,NaN
4,3/26/2019,118.620003,118.709999,116.849998,117.910004,116.370346,26097700,NaN
...,...,...,...,...,...,...,...,...
249,3/16/2020,140.000000,149.350006,135.000000,135.419998,135.419998,87905900,156.138001
250,3/17/2020,140.000000,147.500000,135.000000,146.570007,146.570007,81059800,154.344002
251,3/18/2020,138.000000,146.000000,135.020004,140.399994,140.399994,81593200,151.329001
252,3/19/2020,142.770004,150.149994,139.000000,142.710007,142.710007,85801400,148.973001


In [ ]:
## calculate the Exponential Moving Average and add it
## to a new column in df;
## ewm means exponential moving average
df['EMA'] = df['Close'].ewm(span=long_window).mean()
df

,Date,Open,High,Low,Close,Adj Close,Volume,SMA,EMA
0,3/20/2019,117.389999,118.750000,116.709999,117.519997,115.985443,28113300,NaN,117.519997
1,3/21/2019,117.139999,120.820000,117.089996,120.220001,118.650192,29854400,NaN,118.914999
2,3/22/2019,119.500000,119.589996,117.040001,117.050003,115.521576,33624500,NaN,118.251444
3,3/25/2019,116.559998,118.010002,116.320000,117.660004,116.123627,27067100,NaN,118.088481
4,3/26/2019,118.620003,118.709999,116.849998,117.910004,116.370346,26097700,NaN,118.047873
...,...,...,...,...,...,...,...,...,...
249,3/16/2020,140.000000,149.350006,135.000000,135.419998,135.419998,87905900,156.138001,163.742999
250,3/17/2020,140.000000,147.500000,135.000000,146.570007,146.570007,81059800,154.344002,162.635064
251,3/18/2020,138.000000,146.000000,135.020004,140.399994,140.399994,81593200,151.329001,161.200543
252,3/19/2020,142.770004,150.149994,139.000000,142.710007,142.710007,85801400,148.973001,160.007605


In [ ]:
## calculate the Momentum and add it to a new column in df;
## Momentum is defined as close [today] / close [5 days ago]
momentum_window = 40
df['Momentum'] = df['Close'] / df['Close'].shift(momentum_window)
df

/tmp/ipykernel_3257/1655882467.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Momentum'] = df['Close'] / df['Close'].shift(momentum_window)


,Date,Open,High,Low,Close,Adj Close,Volume,SMA,EMA,Momentum,STD,BB,Percent_Change
10,4/3/2019,119.860001,120.430000,119.150002,119.970001,118.403458,22860700,118.266001,118.293780,NaN,1.193559,0.713831,0.511059
11,4/4/2019,120.099998,120.230003,118.379997,119.360001,117.801422,20112800,118.180001,118.418667,NaN,1.074737,0.548971,-0.442070
12,4/5/2019,119.389999,120.230003,119.370003,119.889999,118.324493,15826200,118.464001,118.582391,NaN,1.113159,0.640518,-0.033354
13,4/8/2019,119.809998,120.019997,118.639999,119.930000,118.363968,15116200,118.691000,118.725649,NaN,1.156663,0.535592,0.544937
14,4/9/2019,118.629997,119.540001,118.580002,119.279999,117.722450,17612000,118.828000,118.782216,NaN,1.137012,0.198766,-0.757137
...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,3/13/2020,147.500000,161.910004,140.729996,158.830002,158.830002,92727400,159.875000,165.696309,0.973342,9.518973,-0.054890,17.286962
249,3/16/2020,140.000000,149.350006,135.000000,135.419998,135.419998,87905900,156.138001,163.742999,0.814949,10.943980,-0.946548,-7.607292
250,3/17/2020,140.000000,147.500000,135.000000,146.570007,146.570007,81059800,154.344002,162.635064,0.877139,10.894856,-0.356774,4.394596
251,3/18/2020,138.000000,146.000000,135.020004,140.399994,140.399994,81593200,151.329001,161.200543,0.843243,10.138429,-0.538989,-1.618676


In [ ]:
## calculate the standard deviation of SMA over a rolling window
## ddof means degree of freedom, set it to zero to get the population std
df['STD'] = df['Close'].rolling(short_window).std(ddof=0)
## calculate the Bollinger Band for each day
## BB > 1 means the closing price is above the upper band
## BB < -1 means the closing price is below the lower band
df['BB'] = (df['Close'] - df['SMA']) / (2 * df['STD'])
df

/tmp/ipykernel_3257/3208499057.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['STD'] = df['Close'].rolling(short_window).std(ddof=0)
/tmp/ipykernel_3257/3208499057.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['BB'] = (df['Close'] - df['SMA']) / (2 * df['STD'])


,Date,Open,High,Low,Close,Adj Close,Volume,SMA,EMA,Momentum,STD,BB,Percent_Change
10,4/3/2019,119.860001,120.430000,119.150002,119.970001,118.403458,22860700,118.266001,118.293780,NaN,NaN,NaN,0.511059
11,4/4/2019,120.099998,120.230003,118.379997,119.360001,117.801422,20112800,118.180001,118.418667,NaN,NaN,NaN,-0.442070
12,4/5/2019,119.389999,120.230003,119.370003,119.889999,118.324493,15826200,118.464001,118.582391,NaN,NaN,NaN,-0.033354
13,4/8/2019,119.809998,120.019997,118.639999,119.930000,118.363968,15116200,118.691000,118.725649,NaN,NaN,NaN,0.544937
14,4/9/2019,118.629997,119.540001,118.580002,119.279999,117.722450,17612000,118.828000,118.782216,NaN,NaN,NaN,-0.757137
...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,3/13/2020,147.500000,161.910004,140.729996,158.830002,158.830002,92727400,159.875000,165.696309,0.973342,9.518973,-0.054890,17.286962
249,3/16/2020,140.000000,149.350006,135.000000,135.419998,135.419998,87905900,156.138001,163.742999,0.814949,10.943980,-0.946548,-7.607292
250,3/17/2020,140.000000,147.500000,135.000000,146.570007,146.570007,81059800,154.344002,162.635064,0.877139,10.894856,-0.356774,4.394596
251,3/18/2020,138.000000,146.000000,135.020004,140.399994,140.399994,81593200,151.329001,161.200543,0.843243,10.138429,-0.538989,-1.618676


In [ ]:
## define the output for the ANN:
## calculate the percent change of closing price for each day
## shift(-1) means yesterday (applies to each element of df)
increase = df['Close'] - df['Close'].shift(-1)
df['Percent_Change'] = (increase / df['Close'].shift(-1))*100
df

/tmp/ipykernel_3257/3688810009.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Percent_Change'] = (increase / df['Close'].shift(-1))*100


,Date,Open,High,Low,Close,Adj Close,Volume,SMA,EMA,Momentum,STD,BB,Percent_Change
10,4/3/2019,119.860001,120.430000,119.150002,119.970001,118.403458,22860700,118.266001,118.293780,NaN,NaN,NaN,0.511059
11,4/4/2019,120.099998,120.230003,118.379997,119.360001,117.801422,20112800,118.180001,118.418667,NaN,NaN,NaN,-0.442070
12,4/5/2019,119.389999,120.230003,119.370003,119.889999,118.324493,15826200,118.464001,118.582391,NaN,NaN,NaN,-0.033354
13,4/8/2019,119.809998,120.019997,118.639999,119.930000,118.363968,15116200,118.691000,118.725649,NaN,NaN,NaN,0.544937
14,4/9/2019,118.629997,119.540001,118.580002,119.279999,117.722450,17612000,118.828000,118.782216,NaN,NaN,NaN,-0.757137
...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,3/13/2020,147.500000,161.910004,140.729996,158.830002,158.830002,92727400,159.875000,165.696309,0.973342,9.518973,-0.054890,17.286962
249,3/16/2020,140.000000,149.350006,135.000000,135.419998,135.419998,87905900,156.138001,163.742999,0.814949,10.943980,-0.946548,-7.607292
250,3/17/2020,140.000000,147.500000,135.000000,146.570007,146.570007,81059800,154.344002,162.635064,0.877139,10.894856,-0.356774,4.394596
251,3/18/2020,138.000000,146.000000,135.020004,140.399994,140.399994,81593200,151.329001,161.200543,0.843243,10.138429,-0.538989,-1.618676


In [ ]:
# remove remove rows that contain non-numerical values:
df = df.dropna(axis=0)
df

,Date,Open,High,Low,Close,Adj Close,Volume,SMA,EMA,Momentum,STD,BB,Percent_Change
50,5/31/2019,124.230003,124.620003,123.320000,123.680000,122.516846,26646800,126.179001,125.596959,1.030924,1.196090,-1.044655,3.204276
51,6/3/2019,123.849998,124.370003,119.010002,119.839996,118.712952,37983600,125.356000,125.213588,1.004021,2.100954,-1.312738,-2.695687
52,6/4/2019,121.279999,123.279999,120.650002,123.160004,122.001740,29382600,125.050001,125.077118,1.027275,2.174388,-0.434604,-2.121909
53,6/5/2019,124.949997,125.870003,124.209999,125.830002,124.646629,24926100,124.943001,125.127054,1.049195,2.105969,0.210592,-1.556875
54,6/6/2019,126.440002,127.970001,125.599998,127.820000,126.617912,21459000,124.958001,125.305344,1.071596,2.125780,0.673165,-2.724501
...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,3/12/2020,145.300003,153.470001,138.580002,139.059998,139.059998,93226400,160.192999,166.169848,0.857707,9.531859,-1.108546,-12.447273
248,3/13/2020,147.500000,161.910004,140.729996,158.830002,158.830002,92727400,159.875000,165.696309,0.973342,9.518973,-0.054890,17.286962
249,3/16/2020,140.000000,149.350006,135.000000,135.419998,135.419998,87905900,156.138001,163.742999,0.814949,10.943980,-0.946548,-7.607292
250,3/17/2020,140.000000,147.500000,135.000000,146.570007,146.570007,81059800,154.344002,162.635064,0.877139,10.894856,-0.356774,4.394596


In [ ]:
# define the inputs of the ANN:
X = np.array(df[['SMA','EMA','Momentum','BB']])
# define the output of the ANN:
Y = np.array(df['Percent_Change'])

In [ ]:
# print the first 10 rows of X
X[:10]

array([[126.1790015 , 125.59695914,   1.03092439,  -1.04465473],
       [125.3560004 , 125.21358834,   1.00402141,  -1.31273789],
       [125.0500007 , 125.07711831,   1.02727504,  -0.43460429],
       [124.9430007 , 125.12705406,   1.04919538,   0.21059224],
       [124.9580009 , 125.30534368,   1.07159625,   0.67316451],
       [125.4800003 , 125.7081663 ,   1.09326892,   1.03072256],
       [126.1160011 , 126.16296091,   1.10196962,   0.90425311],
       [126.7100013 , 126.55417161,   1.09218693,   0.67202969],
       [127.3650016 , 126.87896206,   1.08624537,   0.49128665],
       [128.024002  , 127.23653672,   1.09563642,   0.48795331]])

In [ ]:
# print the first 10 rows of Y
Y[:10]

array([ 3.20427581, -2.69568682, -2.12190889, -1.55687529, -2.72450089,
       -0.90498638,  0.37850112,  0.46391435, -0.62726871, -0.0981427 ])

In [ ]:
## create a model for the ANN
model = keras.Sequential()
# Create ANN with 4 inputs + 3 hidden layers + 1 output layer
## first hidden layer that accepts 4 input features (SMA, EMA, Momentum, BB);
## Dense means every neuron in the layer connects to every neuron in the
## previous layer.
## define input layer:
model.add(keras.layers.Input(shape=(4,)))
## add the first hidden layer with 24 neurons to the ANN:
model.add(keras.layers.Dense(24, activation='relu'))
## add the second hidden layer with 12 neurons to the ANN:
model.add(keras.layers.Dense(12, activation='relu'))
## add the third hidden layer with 6 neurons to the ANN:
model.add(keras.layers.Dense(6, activation='relu'))
## add an output layer with a single output (Percentage_Change):
model.add(keras.layers.Dense(1, activation='linear'))

## set the optimization algorithm used for minimizing loss function
## use gradient descent (adam) to minimize error (loss)
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_11 (Dense)                │ (None, 24)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 12)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 6)              │            78 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │             7 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 505 (1.97 KB)

 Trainable params: 505 (1.97 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# print the initial values of weights:
weights = model.get_weights()
print_weights(weights)


******* WEIGHTS OF ANN *******

Weights W0:
 [[ 0.45887518 -0.28397804  0.30117267 -0.41861507  0.17747283 -0.39883983
  -0.21964715 -0.45290434  0.28042614 -0.40822893  0.18589592  0.20193928
  -0.16045469  0.18602109 -0.09808853 -0.13439178  0.1425708  -0.35715988
  -0.09047469  0.2712028  -0.09424284  0.1454407   0.02353275 -0.36254808]
 [-0.21231806 -0.05273858 -0.05016559  0.0098736   0.02422154  0.459449
  -0.03209817 -0.08997297  0.00715286  0.0489347   0.40101337 -0.04689875
   0.171143    0.15031105 -0.37580353  0.35669732 -0.32650486 -0.03068814
  -0.24416052  0.31822658 -0.3565116   0.4192341   0.00155452 -0.37650687]
 [-0.17823803  0.26151067 -0.09266603  0.4479193   0.09637213  0.17475939
  -0.42823106 -0.17357692 -0.10833341 -0.314088   -0.3242644   0.10693818
  -0.19999743  0.10206163 -0.3507519  -0.15318894  0.24907792  0.22039765
  -0.22097652  0.0800367   0.40986377 -0.02024198 -0.23723194 -0.38889363]
 [ 0.13160759  0.16850108 -0.06287992  0.1331709  -0.32550317 -0.

In [ ]:
## train the ANN model using 50 iterations
model.fit(X, Y, epochs=50, verbose=0)

print('\n\n********** ANN training complete **********\n\n')



********** ANN training complete **********




In [ ]:
# print the weighst after training:
weights = model.get_weights();
print_weights(weights)


******* WEIGHTS OF ANN *******

Weights W0:
 [[ 0.4528499  -0.28397804  0.3120024  -0.41861507  0.20525204 -0.37613308
  -0.21964715 -0.45290434  0.2583211  -0.40822893  0.20213814  0.22318575
  -0.17118849  0.16909133 -0.09808853 -0.15787911  0.1425708  -0.35715988
  -0.09047469  0.2509801  -0.09424284  0.12619373  0.00432364 -0.36254808]
 [-0.21830396 -0.05273858 -0.03935487  0.0098736   0.05199829  0.482131
  -0.03209817 -0.08997297 -0.01489024  0.0489347   0.41724384 -0.02568372
   0.16037492  0.13339315 -0.37580353  0.333212   -0.32650486 -0.03068814
  -0.24416052  0.2980208  -0.3565116   0.3999974  -0.01770062 -0.37650687]
 [-0.1841388   0.26151067 -0.08185863  0.4479193   0.12417094  0.19751234
  -0.42823106 -0.17357692 -0.13066861 -0.314088   -0.30797866  0.12832795
  -0.21065901  0.0850477  -0.3507519  -0.17667887  0.24907792  0.22039765
  -0.22097652  0.05982968  0.40986377 -0.03954767 -0.25628772 -0.38889363]
 [ 0.12161492  0.16850108 -0.052208    0.1331709  -0.30208784 -0.

In [ ]:
#% ANN PREDICTION

## insert the inputs for the latest trading day into an array
latest_SMA = df.iloc[-1]['SMA']
latest_EMA = df.iloc[-1]['EMA']
latest_Momentum = df.iloc[-1]['Momentum']
latest_BB = df.iloc[-1]['BB']
latest_inputs = np.array([[latest_SMA, latest_EMA,latest_Momentum,latest_BB]])

prediction = model.predict(latest_inputs)[0,0]
latest_percent_change = df.iloc[-1]['Percent_Change']
print("\ntarget value for percent change: ",latest_percent_change)
print('\n***************************************')
print('ANN Predicted Next Day Stock Movement: %+.2f%%' % (prediction))
print('***************************************')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step

target value for percent change:  -1.6186762572298088

***************************************
ANN Predicted Next Day Stock Movement: -0.04%
***************************************
